## Grad CAM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import cv2 as cv
import torch

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

import glob

from modelling.datasets import ImageDataset
from modelling.network import CNNDummy

In [ ]:
model_path='pot-plant-classifier.pth'

device=torch.device('mps' if torch.backends.mps.is_built()
                    else 'cuda' if torch.cuda.is_available()
                    else 'cpu')

model=CNNDummy(num_classes=4)
torch.save(model.state_dict(), model_path)
model.load_state_dict(torch.load(model_path, weights_only=False))

model.to(device)
model.eval()

In [ ]:
base_path='./data/pot_plants'

plant_names=['rudo', 'baya', 'greg', 'yuki']
num_classes=len(plant_names)

# Load the data
test_paths=glob.glob(base_path+"/train_valid/*.jpg")
test_data=ImageDataset(test_paths, class_names=plant_names)

In [ ]:
# Get random instance
image, target=test_data.__getitem__(3)

# Format input
input_img=image.unsqueeze(0).to(device)

# Format target
# argmax(): when the target is one-hot vector
target=torch.argmax(target).item()
target_name=plant_names[target]

# Get the prediction
output=model(input_img)
pred=torch.argmax(output).item()
pred_name=plant_names[pred]

# Display prediction
rgb_image=image.permute(1, 2, 0).numpy()
plt.imshow(rgb_image)
plt.title(f"Target: {target_name} ({target})\nPred: {pred_name} ({pred})")
plt.axis('off')

In [ ]:
def get_canny_edge(img, threshold1=30, threshold2=80):
    # Gray scale the image
    gray=cv.cvtColor(img, cv.COLOR_RGB2GRAY)
    gray=gray*255
    gray=gray.astype(np.uint8)

    # Gaussian Blur
    gray=cv.GaussianBlur(gray, (5, 5), 0)

    # Get the edge
    edge=255-cv.Canny(gray, threshold1, threshold2)
    edge=np.stack([edge]*3, axis=-1)/255

    return edge

fig, ax=plt.subplots(1, 2, figsize=(10, 6))
ax[0].imshow(rgb_image)
ax[0].set_title("Original Image")

edge=get_canny_edge(rgb_image)
ax[1].imshow(edge)
ax[1].set_title('Canny edge')

for a in ax:
    a.set_xticks([])
    a.set_yticks([])

In [ ]:
def plot_gradcam(rgb_image, visualization, title='Input'):
    fig, ax=plt.subplots(1, 2, figsize=(10, 5))

    ax[0].imshow(rgb_image)
    ax[0].set_title(title)

    ax[1].imshow(visualization)
    ax[1].set_title('Grad-Cam Heatmap')

    for a in ax:
        a.set_xticks([])
        a.set_yticks([])

In [ ]:
# Get the target layer and class of interest
classes=[ClassifierOutputTarget(1)]
layers=[model.conv_layers[6]]

In [ ]:
# Get the Gradcam heatmap
cam=GradCAM(model=model, target_layers=layers)
heatmap=cam(input_tensor=image.unsqueeze(0), targets=classes)

print(heatmap.shape)
plt.imshow(heatmap[0])
plt.axis('off')

In [ ]:
# Plot the heatmap
edge=get_canny_edge(rgb_image)
visualization=show_cam_on_image(edge, heatmap[0], use_rgb=True)

plot_gradcam(rgb_image, visualization)

### Multiple Layers

In [ ]:
# All layers
layers=[model.conv_layers[0], model.conv_layers[3], model.conv_layers[6]]

In [ ]:
cam=GradCAM(model=model, target_layers=layers)
heatmap=cam(input_tensor=image.unsqueeze(0), targets=classes)

print(heatmap.shape)
plt.imshow(heatmap[0])
plt.axis('off')

In [ ]:
# Plot the heatmap
edge=get_canny_edge(rgb_image)
visualization=show_cam_on_image(edge, heatmap[0], use_rgb=True)

plot_gradcam(rgb_image, visualization)

In [ ]:
# Get seperate visualizations for each layer
maps=[]

for layer in layers:
    cam=GradCAM(model=model, target_layers=[layer])
    heatmap=cam(input_tensor=image.unsqueeze(0), targets=classes)
    visualization=show_cam_on_image(edge, heatmap[0], use_rgb=True)
    maps.append(visualization)

fig, ax=plt.subplots(1, 3, figsize=(15, 5))

for i, vis in enumerate(maps):
    ax[i].imshow(vis)
    ax[i].set_title(f'Conv {i+1}')
    ax[i].set_xticks([])
    ax[i].set_yticks([])

### Multiple Classes

In [ ]:
classes=[ClassifierOutputTarget(0),
         ClassifierOutputTarget(1),
         ClassifierOutputTarget(2),
         ClassifierOutputTarget(3)]

cam=GradCAM(model=model, target_layers=layers)
heatmap=cam(input_tensor=image.unsqueeze(0), targets=classes)

edge=get_canny_edge(rgb_image)
visualization=show_cam_on_image(edge, heatmap[0], use_rgb=True)
plot_gradcam(rgb_image=rgb_image, visualization=visualization)

In [ ]:
print(output)